# CSIT 595 Assignment 1: modular Fashion-MNIST comparison

This notebook imports the model implementations from the Python modules in this folder.
Set `CONFIG["full_run"] = False` for a smoke test or `True` for the final GPU experiment.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

from assignment1_common import (
    CONFIG, DEVICE, OUT, EvaluatorCNN, configure_run, evaluator_accuracy,
    predicted_histogram, show_grid, synchronize_cuda, test_loader, train_evaluator, evaluate_model,
)
from assignment1_vae import VAE, train_vae, sample_vae
from assignment1_gan import Generator, Discriminator, train_gan, sample_gan
from assignment1_diffusion import TinyUNet, train_diffusion, sample_diffusion

# Change this one flag before running. The model modules contain the implementation.
CONFIG["full_run"] = True
configure_run(CONFIG["full_run"])


def run_experiment():
    print("Training evaluator...")
    evaluator = EvaluatorCNN().to(DEVICE)
    evaluator, evaluator_history = train_evaluator(evaluator)
    print("Evaluator test accuracy:", evaluator_accuracy(evaluator))

    print("Training VAE...")
    vae = VAE(latent_dim=64).to(DEVICE)
    vae, vae_history, vae_seconds = train_vae(vae)

    print("Training GAN...")
    generator = Generator(latent_dim=128).to(DEVICE)
    discriminator = Discriminator().to(DEVICE)
    gan_checkpoints = []

    def record_gan_checkpoint(epoch, current_generator):
        current_generator.eval()
        with torch.no_grad():
            samples = current_generator(torch.randn(CONFIG["metric_count"], current_generator.latent_dim, device=DEVICE))
        histogram = predicted_histogram(samples, evaluator).tolist()
        gan_checkpoints.append({"epoch": epoch, **{f"class_{i}": value for i, value in enumerate(histogram)}})
        current_generator.train()

    generator, discriminator, gan_history, gan_seconds = train_gan(
        generator, discriminator, checkpoint_callback=record_gan_checkpoint
    )

    print("Training diffusion model...")
    denoiser = TinyUNet(base_channels=32, time_dim=64).to(DEVICE)
    denoiser, diffusion, diffusion_history, diffusion_seconds = train_diffusion(denoiser)

    vae_metrics, _ = evaluate_model(
        "VAE", lambda count: sample_vae(vae, count), evaluator,
        training_seconds=vae_seconds, parameters=sum(p.numel() for p in vae.parameters()),
    )
    gan_metrics, _ = evaluate_model(
        "GAN", lambda count: sample_gan(generator, count), evaluator,
        training_seconds=gan_seconds,
        parameters=sum(p.numel() for p in generator.parameters()) + sum(p.numel() for p in discriminator.parameters()),
    )
    ddpm_metrics, _ = evaluate_model(
        "DDPM", lambda count: sample_diffusion(diffusion, denoiser, count, steps=CONFIG["diffusion_steps"]), evaluator,
        training_seconds=diffusion_seconds, parameters=sum(p.numel() for p in denoiser.parameters()),
    )
    results = pd.DataFrame([vae_metrics, gan_metrics, ddpm_metrics])
    results.to_csv(OUT / "results.csv", index=False)
    pd.DataFrame(gan_checkpoints).to_csv(OUT / "gan_checkpoint_histograms.csv", index=False)
    pd.DataFrame(evaluator_history).to_csv(OUT / "evaluator_history.csv", index=False)
    pd.DataFrame(vae_history).to_csv(OUT / "vae_history.csv", index=False)
    pd.DataFrame(gan_history).to_csv(OUT / "gan_history.csv", index=False)
    pd.DataFrame(diffusion_history).to_csv(OUT / "diffusion_history.csv", index=False)

    history_fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    evaluator_frame = pd.DataFrame(evaluator_history)
    axes[0, 0].plot(evaluator_frame["epoch"], evaluator_frame["loss"], label="loss")
    axes[0, 0].set_title("Evaluator loss")
    vae_frame = pd.DataFrame(vae_history)
    axes[0, 1].plot(vae_frame["epoch"], vae_frame["recon_loss"], label="reconstruction")
    axes[0, 1].plot(vae_frame["epoch"], vae_frame["kl"], label="KL")
    axes[0, 1].plot(vae_frame["epoch"], vae_frame["total_loss"], label="total")
    axes[0, 1].set_title("VAE losses")
    axes[0, 1].legend()
    gan_frame = pd.DataFrame(gan_history)
    axes[1, 0].plot(gan_frame["epoch"], gan_frame["g_loss"], label="generator")
    axes[1, 0].plot(gan_frame["epoch"], gan_frame["d_loss"], label="discriminator")
    axes[1, 0].set_title("GAN losses")
    axes[1, 0].legend()
    diffusion_frame = pd.DataFrame(diffusion_history)
    axes[1, 1].plot(diffusion_frame["step"], diffusion_frame["mse"], marker="o")
    axes[1, 1].set_title("Diffusion noise-prediction MSE")
    for axis in axes.flat:
        axis.grid(alpha=0.25)
    history_fig.tight_layout()
    history_fig.savefig(OUT / "training_curves.png", dpi=160)
    plt.close(history_fig)
    print(results)

    checkpoint_frame = pd.DataFrame(gan_checkpoints).set_index("epoch")
    checkpoint_frame.plot(kind="bar", stacked=True, figsize=(10, 5), ylim=(0, 1), title="GAN evaluator-label histogram by checkpoint")
    plt.tight_layout()
    plt.savefig(OUT / "gan_mode_collapse_checkpoints.png", dpi=160)
    plt.close()

    diffusion_speed_rows = []
    for sampling_steps in (25, 100, CONFIG["diffusion_steps"]):
        synchronize_cuda()
        started = __import__("time").perf_counter()
        diffusion_samples = sample_diffusion(diffusion, denoiser, CONFIG["metric_count"], steps=sampling_steps)
        synchronize_cuda()
        elapsed = __import__("time").perf_counter() - started
        labels, features = __import__("assignment1_common").evaluator_outputs(diffusion_samples, evaluator)
        real_features, real_labels = __import__("assignment1_common").reference_features(CONFIG["sample_count"], evaluator)
        speed_metrics = __import__("assignment1_common").coverage_metrics(real_features, real_labels, features.numpy(), labels.numpy())
        speed_metrics.update({
            "sampling_steps": sampling_steps,
            "images_per_second": CONFIG["metric_count"] / elapsed,
            "time_for_5000_images": 5000 * elapsed / CONFIG["metric_count"],
        })
        diffusion_speed_rows.append(speed_metrics)
        show_grid(diffusion_samples[:CONFIG["sample_count"]], f"DDPM samples ({sampling_steps} steps)", f"diffusion_samples_{sampling_steps}_steps.png")
    pd.DataFrame(diffusion_speed_rows).drop(columns="generated_histogram").to_csv(OUT / "diffusion_speed_quality.csv", index=False)

    preview, _ = next(iter(test_loader))
    preview = preview[:10].to(DEVICE)
    with torch.no_grad():
        reconstruction, _, _ = vae(preview)
        z0, z1 = torch.randn(1, vae.latent_dim, device=DEVICE), torch.randn(1, vae.latent_dim, device=DEVICE)
        vae_interp = torch.cat([vae.decode(torch.lerp(z0, z1, f.item())) for f in torch.linspace(0, 1, 8)])
        z0, z1 = torch.randn(1, generator.latent_dim, device=DEVICE), torch.randn(1, generator.latent_dim, device=DEVICE)
        gan_interp = torch.cat([generator(torch.lerp(z0, z1, f.item())) for f in torch.linspace(0, 1, 8)]).cpu()
    show_grid(torch.cat([preview.cpu(), reconstruction.cpu()]), "VAE originals and reconstructions", "vae_reconstruction.png")
    show_grid(vae_interp.cpu(), "VAE latent interpolation", "vae_interpolation.png", columns=8)
    show_grid(gan_interp, "GAN latent interpolation", "gan_interpolation.png", columns=8)
    show_grid(sample_vae(vae, CONFIG["sample_count"]), "VAE prior samples", "vae_samples.png")
    show_grid(sample_gan(generator, CONFIG["sample_count"]), "GAN samples", "gan_samples.png")
    show_grid(sample_diffusion(diffusion, denoiser, CONFIG["sample_count"], steps=min(CONFIG["diffusion_steps"], 100)), "DDPM samples", "diffusion_samples.png")
    print("Saved artifacts to:", OUT)


if __name__ == "__main__":
    run_experiment()
